In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [8]:
train = pd.read_csv("../data/splits/train.csv")
validation = pd.read_csv("../data/splits/validation.csv")
test = pd.read_csv("../data/splits/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (5947, 5)
Validation: (1278, 5)
Test: (1271, 5)


In [10]:
features = ["icare_iop", "cct"]
target = "applanation_iop"

X_train = train[features]
y_train = train[target]

X_validation = validation[features]
y_validation = validation[target]

X_test = test[features]
y_test = test[target]

In [13]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [ ]:
model.fit(X_train, y_train)

In [15]:
validation_predictions = model.predict(X_validation)

In [16]:
validation_mae = mean_absolute_error(
    y_validation,
    validation_predictions
)

validation_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        validation_predictions
    )
)

validation_r2 = r2_score(
    y_validation,
    validation_predictions
)

print("Validation Results")
print("------------------")
print(f"MAE:  {validation_mae:.3f} mmHg")
print(f"RMSE: {validation_rmse:.3f} mmHg")
print(f"R²:   {validation_r2:.3f}")

Validation Results
------------------
MAE:  3.303 mmHg
RMSE: 4.452 mmHg
R²:   0.679


In [ ]:
# FINAL TEST EVALUATION

## Random Forest Hyperparameter Tuning
### Experiment 1: Number of Trees - 50
### Experiment 2: Max Depth - 6

In [22]:
max_depth_values = [3, 4, 5, 6, 7, 8, 9]

results = []

for depth in max_depth_values:

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=depth,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)

    mae = mean_absolute_error(y_validation, predictions)
    rmse = np.sqrt(mean_squared_error(y_validation, predictions))
    r2 = r2_score(y_validation, predictions)

    results.append({
        "max_depth": depth,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df

,max_depth,MAE,RMSE,R2
0,3,2.927977,4.027350,0.737011
1,4,2.832431,3.923085,0.750452
2,5,2.822080,3.907379,0.752446
3,6,2.824064,3.899894,0.753393
4,7,2.837363,3.902997,0.753001
5,8,2.858816,3.919899,0.750857
6,9,2.883647,3.950576,0.746942


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    results_df["max_depth"],
    results_df["MAE"],
    marker="o"
)

plt.xlabel("Max Depth")
plt.ylabel("Validation MAE (mmHg)")
plt.title("Effect of Max Depth on Random Forest Performance")

plt.grid(True, alpha=0.3)
plt.show()

### Experiment 3: Minimum Samples per Leaf

In [23]:
leaf_values = [1, 2, 3, 5, 10, 15, 20]

leaf_results = []

for leaf in leaf_values:

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=6,
        min_samples_leaf=leaf,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)

    mae = mean_absolute_error(y_validation, predictions)
    rmse = np.sqrt(mean_squared_error(y_validation, predictions))
    r2 = r2_score(y_validation, predictions)

    leaf_results.append({
        "min_samples_leaf": leaf,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

leaf_results_df = pd.DataFrame(leaf_results)

leaf_results_df

,min_samples_leaf,MAE,RMSE,R2
0,1,2.824064,3.899894,0.753393
1,2,2.828582,3.912917,0.751744
2,3,2.827304,3.906244,0.752590
3,5,2.820271,3.910770,0.752016
4,10,2.819424,3.911634,0.751906
5,15,2.822433,3.919220,0.750943
6,20,2.823888,3.917055,0.751218


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    leaf_results_df["min_samples_leaf"],
    leaf_results_df["MAE"],
    marker="o"
)

plt.xlabel("Minimum Samples per Leaf")
plt.ylabel("Validation MAE (mmHg)")
plt.title("Effect of Minimum Samples per Leaf")

plt.grid(True, alpha=0.3)
plt.show()

### Experiment 4: Minimum Samples Required to Split

In [25]:
split_values = [2, 5, 10, 20, 30, 50]

split_results = []

for split in split_values:

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=6,
        min_samples_leaf=1,
        min_samples_split=split,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)

    mae = mean_absolute_error(y_validation, predictions)
    rmse = np.sqrt(mean_squared_error(y_validation, predictions))
    r2 = r2_score(y_validation, predictions)

    split_results.append({
        "min_samples_split": split,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

split_results_df = pd.DataFrame(split_results)

split_results_df

,min_samples_split,MAE,RMSE,R2
0,2,2.824064,3.899894,0.753393
1,5,2.824576,3.904743,0.752780
2,10,2.823756,3.904264,0.752840
3,20,2.817644,3.891856,0.754409
4,30,2.815032,3.895298,0.753974
5,50,2.816511,3.908888,0.752255


In [ ]:
### Experiment 5: Combined Hyperparameter Search